# Chapter 15 – Recurrences: RecursiveSum, Tower of Hanoi and closed forms (Java)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algorithms/blob/main/notebooks/ch15_recurrences_java.ipynb)

Numerical check of the examples of the chapter: the algorithm **RecursiveSum** with the count of the steps $S(n)$ (which satisfies $S(1)=2$, $S(n)=S(n-1)+3$, that is $S(n)=3n-1$), the **Tower of Hanoi**, which counts the moves ($2^n-1$), and the comparison between some recurrences of the chapter, computed directly, and their closed form.

The full chapter, with pseudo-code, proofs and worked examples, is [on the website](https://fabiofurini.github.io/algorithms/analysis-recursion/15-recurrences/).

Each program is written to a `.java` file by the `%%writefile` cell and then compiled and run with `javac` and `java`: run the cells in order, from top to bottom (the first cell checks or installs the JDK). No external libraries.

**Indices.** In the pseudo-code of the notes arrays are indexed from 1; in the code (as in all the languages used here) from 0: the element `a[i]` of the pseudo-code is the element of index `i-1` in the code, and a range `a[l..r]` becomes `a[l-1..r-1]`. The printed output of the notebooks uses indices from 0.

**Names.** The functions have the name of the pseudo-code, in the convention of the language (as in the introductory chapters): `RecursiveSum` → `recursiveSum` (camelCase).

In [1]:
# Check (and if needed install) the JDK
!java -version || apt-get -qq install -y default-jdk-headless

openjdk version "25.0.4.1" 2026-08-18
OpenJDK Runtime Environment (build 25.0.4.1+1-1-26.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 25.0.4.1+1-1-26.04.4-Ubuntu, mixed mode, sharing)


## 1. RecursiveSum and the count of the steps

**RecursiveSum**$(a,n)$ — *Input*: an array $a[1..n]$ of $n\ge1$ integers. *Output*: the sum $a[1]+\dots+a[n]$.

The code follows the pseudo-code (lines 2, 3, 5, 6); a counter `steps` adds up the steps executed by every call, as in the chapter: 2 steps if $n=1$, 3 steps plus the recursive call if $n>1$. The second function is `sumRecDecrease` of the code of the course, which stops at the empty array ($n=0$) and therefore makes one more call.

In [2]:
%%writefile RecursiveSum.java
import java.util.*;

public class RecursiveSum {
    static long steps = 0;   // steps executed (lines of pseudo-code)
    static int calls = 0;    // number of calls

    // Sum of a[0..n-1] (pseudo-code: a[1..n]), n >= 1
    static int recursiveSum(int[] a, int n) {
        calls++;
        steps++;                  // line 2: if n = 1
        if (n == 1) {
            steps++;              // line 3: return a[1]
            return a[0];
        }
        steps += 2;               // lines 5 and 6
        int s = recursiveSum(a, n - 1);
        return s + a[n - 1];
    }

    // Variant of the course code: base case n = 0
    static long sumRecDecrease(int[] a, int n) {
        calls++;
        if (n == 0) {
            return 0;
        }
        return sumRecDecrease(a, n - 1) + a[n - 1];
    }

    public static void main(String[] args) {
        int[] a = {3, 1, 4, 1, 5, 9, 2, 6};
        int n = 8;
        System.out.println("a = " + Arrays.toString(a));
        calls = 0;
        int r = recursiveSum(a, n);
        System.out.println("RecursiveSum(a, 8) = " + r + " with " + calls + " calls");
        calls = 0;
        long r2 = sumRecDecrease(a, n);
        System.out.println("sumRecDecrease(a, 8) = " + r2 + " with " + calls + " calls");
        System.out.println("n  measured S(n)  3n-1");
        for (int k = 1; k <= 8; k++) {
            steps = 0;
            recursiveSum(a, k);
            System.out.printf("%-2d %12d %5d%n", k, steps, 3 * k - 1);
        }
    }
}

Writing RecursiveSum.java


In [3]:
!javac RecursiveSum.java && java RecursiveSum

a = [3, 1, 4, 1, 5, 9, 2, 6]
RecursiveSum(a, 8) = 31 with 8 calls
sumRecDecrease(a, 8) = 31 with 9 calls
n  measured S(n)  3n-1
1             2     2
2             5     5
3             8     8
4            11    11
5            14    14
6            17    17
7            20    20
8            23    23


**Try it yourself.** Change the array `a` (also its length) and check that the measured steps are always $3n-1$.

In [4]:
%%writefile ProvaRecursiveSum.java
import java.util.*;

public class ProvaRecursiveSum {
    public static void main(String[] args) {
        // Change the array and check that the measured steps are always 3n-1.
        int[] a = {7, -2, 5, 0, 11};
        int n = 5;
        RecursiveSum.steps = 0;
        System.out.println(RecursiveSum.recursiveSum(a, n) + " " + RecursiveSum.steps + " " + (3 * n - 1));
    }
}

Writing ProvaRecursiveSum.java


In [5]:
!javac ProvaRecursiveSum.java && java ProvaRecursiveSum

21 14 14


## 2. Tower of Hanoi: $T(n)=2T(n-1)+1$

**Hanoi**$(n, from, to, aux)$ — *Input*: the number of disks $n\ge1$ and the three pegs. *Output*: the sequence of moves that transfers disks $1,\dots,n$ from *from* to *to*.

The code counts the moves (and, for $n=3$, prints them): the chapter shows that they are exactly $2^n-1$. The initial call is `Hanoi(n, 'A', 'C', 'B')`.

In [6]:
%%writefile Hanoi.java
import java.util.*;

public class Hanoi {
    static long moves = 0;    // number of moves
    static boolean printMoves = false;

    static void hanoi(int n, char from, char to, char aux) {
        if (n == 1) {
            moves++;                                // move disk 1
            if (printMoves) System.out.println("  disk 1: " + from + " -> " + to);
            return;
        }
        hanoi(n - 1, from, aux, to);
        moves++;                                    // move disk n
        if (printMoves) System.out.println("  disk " + n + ": " + from + " -> " + to);
        hanoi(n - 1, aux, to, from);
    }

    public static void main(String[] args) {
        System.out.println("Moves for n = 3:");
        printMoves = true;
        hanoi(3, 'A', 'C', 'B');
        printMoves = false;
        System.out.println("n   moves  2^n-1");
        int[] ns = {1, 2, 3, 4, 5, 6, 7, 8, 10};
        for (int n : ns) {
            moves = 0;
            hanoi(n, 'A', 'C', 'B');
            System.out.printf("%-3d %5d %5d%n", n, moves, (1L << n) - 1);
        }
    }
}

Writing Hanoi.java


In [7]:
!javac Hanoi.java && java Hanoi

Moves for n = 3:
  disk 1: A -> C
  disk 2: A -> B
  disk 1: C -> B
  disk 3: A -> C
  disk 1: B -> A
  disk 2: B -> C
  disk 1: A -> C
n   moves  2^n-1
1       1     1
2       3     3
3       7     7
4      15    15
5      31    31
6      63    63
7     127   127
8     255   255
10   1023  1023


**Try it yourself.** Change the number of disks $n$: the moves grow as $2^n-1$ (with $n=15$ they are already $32\,767$).

In [8]:
%%writefile ProvaHanoi.java
import java.util.*;

public class ProvaHanoi {
    public static void main(String[] args) {
        // Try with more disks: how many moves are needed for n = 15?
        int n = 15;
        Hanoi.moves = 0;
        Hanoi.hanoi(n, 'A', 'C', 'B');
        System.out.println(n + " " + Hanoi.moves + " " + ((1L << n) - 1));
    }
}

Writing ProvaHanoi.java


In [9]:
!javac ProvaHanoi.java && java ProvaHanoi

15 32767 32767


## 3. Recurrences computed directly and closed forms

Every recurrence is computed by a recursive function that follows the definition and compared with the closed form obtained in the chapter (expansion method). For the recurrences with $n/2$ the values of $n$ are powers of 2.

| recurrence | closed form |
|---|---|
| $T(n)=T(n-1)+n$, $T(1)=1$ | $n(n+1)/2$ |
| $S(n)=S(n-1)+3$, $S(1)=2$ (RecursiveSum) | $3n-1$ |
| $T(n)=2T(n-1)+1$, $T(1)=1$ (Hanoi) | $2^n-1$ |
| $T(n)=T(n/2)+1$, $T(1)=1$ | $\log_2 n+1$ |
| $T(n)=2T(n/2)+n$, $T(1)=1$ | $n\log_2 n+n$ |

In [10]:
%%writefile Recurrences.java
import java.util.*;

public class Recurrences {
    static long tQuad(long n)  { return n == 1 ? 1 : tQuad(n - 1) + n; }
    static long tSum(long n)   { return n == 1 ? 2 : tSum(n - 1) + 3; }
    static long tHanoi(long n) { return n == 1 ? 1 : 2 * tHanoi(n - 1) + 1; }
    static long tLog(long n)   { return n == 1 ? 1 : tLog(n / 2) + 1; }
    static long tNLogN(long n) { return n == 1 ? 1 : 2 * tNLogN(n / 2) + n; }

    static long log2(long n) {   // n is a power of 2
        return 63 - Long.numberOfLeadingZeros(n);
    }

    static void confronta(String nome, long[] valori, java.util.function.LongUnaryOperator ric, java.util.function.LongUnaryOperator chiusa) {
        System.out.println(nome);
        for (long n : valori) {
            long r = ric.applyAsLong(n), c = chiusa.applyAsLong(n);
            System.out.printf("  n=%-7d %12d %12d  %s%n", n, r, c, r == c ? "OK" : "DIFFERENT");
        }
    }

    public static void main(String[] args) {
        confronta("T(n) = T(n-1) + n  ->  n(n+1)/2", new long[]{1, 2, 3, 4, 10, 100}, Recurrences::tQuad, n -> n * (n + 1) / 2);
        confronta("S(n) = S(n-1) + 3  ->  3n-1", new long[]{1, 2, 3, 4, 100}, Recurrences::tSum, n -> 3 * n - 1);
        confronta("T(n) = 2T(n-1) + 1  ->  2^n-1", new long[]{1, 2, 3, 4, 10, 20}, Recurrences::tHanoi, n -> (1L << n) - 1);
        confronta("T(n) = T(n/2) + 1  ->  log2(n)+1", new long[]{1, 2, 4, 8, 1024, 1 << 20}, Recurrences::tLog, n -> log2(n) + 1);
        confronta("T(n) = 2T(n/2) + n  ->  n log2(n) + n", new long[]{1, 2, 4, 8, 1024, 1 << 20}, Recurrences::tNLogN, n -> n * log2(n) + n);
    }
}

Writing Recurrences.java


In [11]:
!javac Recurrences.java && java Recurrences

T(n) = T(n-1) + n  ->  n(n+1)/2
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  6            6  OK
  n=4                 10           10  OK
  n=10                55           55  OK
  n=100             5050         5050  OK
S(n) = S(n-1) + 3  ->  3n-1
  n=1                  2            2  OK
  n=2                  5            5  OK
  n=3                  8            8  OK
  n=4                 11           11  OK
  n=100              299          299  OK
T(n) = 2T(n-1) + 1  ->  2^n-1
  n=1                  1            1  OK
  n=2                  3            3  OK
  n=3                  7            7  OK
  n=4                 15           15  OK
  n=10              1023         1023  OK
  n=20           1048575      1048575  OK
T(n) = T(n/2) + 1  ->  log2(n)+1
  n=1                  1            1  OK
  n=2                  2            2  OK
  n=4                  3            3  OK
  n=8                  4            4

**Try it yourself.** Add a recurrence of your own (here $T(n)=T(n-1)+2n-1$, $T(1)=1$) and compare it with its closed form.

In [12]:
%%writefile ProvaRecurrences.java
import java.util.*;

public class ProvaRecurrences {
    static long tMia(long n) { return n == 1 ? 1 : tMia(n - 1) + 2 * n - 1; }

    public static void main(String[] args) {
        // Add a recurrence: T(n) = T(n-1) + 2n - 1 with T(1) = 1 has closed form n^2.
        Recurrences.confronta("T(n) = T(n-1) + 2n - 1  ->  n^2", new long[]{1, 2, 3, 10, 100}, ProvaRecurrences::tMia, n -> n * n);
    }
}

Writing ProvaRecurrences.java


In [13]:
!javac ProvaRecurrences.java && java ProvaRecurrences

T(n) = T(n-1) + 2n - 1  ->  n^2
  n=1                  1            1  OK
  n=2                  4            4  OK
  n=3                  9            9  OK
  n=10               100          100  OK
  n=100            10000        10000  OK
